# PolERo – run 6 on Colab free T4
Runtime → Change runtime type → **T4 GPU**. Add a Colab secret (🔑 left sidebar) named `KAGGLE_API_TOKEN` with your Kaggle token and enable notebook access.
Then Runtime → Run all. Every fold is saved to Google Drive straight away; if Colab disconnects, reconnect and Run all again: finished folds are skipped.

In [ ]:
# 1. Run 6 config, from Rares in issue #2 (17:18)
RUN = "run6"
NAME = "r6-xlmr-large-lr1e-5-s45"
ARGS = ["--model", "FacebookAI/xlm-roberta-large", "--lr", "1e-5",
        "--epochs", "10", "--max_len", "512", "--q_len", "128", "--bs", "8", "--seed", "45"]
FOLDS = [0, 1, 2, 3, 4]

In [ ]:
# 2. GPU check + Google Drive for outputs
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
OUT = f"/content/drive/MyDrive/polero/{RUN}"
import os; os.makedirs(OUT, exist_ok=True); print(OUT)

In [ ]:
# 3. Code + data from the private Kaggle dataset (same files the Kaggle runs use)
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
!pip -q install -U kaggle six sentencepiece
!mkdir -p /content/polero/data
!kaggle datasets download lucazanotti/polero-data -p /content/polero --unzip -o
%cd /content/polero
!mv -f *.csv data/ 2>/dev/null; ls; ls data

In [ ]:
# 4. Train each fold; skip folds already saved in Drive
import subprocess, sys, time
for f in FOLDS:
    tag = f"{NAME}_p0_f{f}_s{ARGS[ARGS.index('--seed')+1]}"
    if os.path.exists(f"{OUT}/{tag}.npz"):
        print("skip", tag); continue
    t = time.time()
    with open(f"{OUT}/{tag}.log", "w") as log:
        rc = subprocess.run([sys.executable, "finetune.py", "--name", NAME, *ARGS, "--fold", str(f), "--out", OUT],
                            stdout=log, stderr=subprocess.STDOUT).returncode
    print(f"{tag}: {'ok' if rc == 0 else f'FAILED ({rc})'} in {(time.time()-t)/60:.1f} min", flush=True)
    if rc: print(open(f"{OUT}/{tag}.log").read()[-2000:])

In [ ]:
# 5. What's done (download the .npz + .log files from Drive/polero/<RUN> when all 5 are there)
!ls -la {OUT}